In [1]:
#Import the requisite library
import spacy

#Sample text
text = "This is a sample number 555-5555."

#Build upon the spaCy Small Model
nlp = spacy.blank("en")

#Create the Ruler and Add it
ruler = nlp.add_pipe("entity_ruler")


patterns = [
                {"label": "PHONE_NUMBER", "pattern": [{"SHAPE": "ddd"},
                {"ORTH": "-", "OP": "?"}, {"SHAPE": "dddd"}]}
            ]
#add patterns to ruler
ruler.add_patterns(patterns)

#create the doc
doc = nlp(text)

#extract entities
for ent in doc.ents:
    print (ent.text, ent.label_)

555-5555 PHONE_NUMBER


```pattern": [{"SHAPE": "ddd"},{"ORTH": "-", "OP": "?"}, {"SHAPE": "dddd"}]}]```

```shape ddd``` - pattern starts with 3 digits
```ORTH - , OP ?``` hyphen is optional maybe or not be present

```shape dddd``` followed by 4 digits

            


# Implementation in RegEx

In [2]:
import re
pattern = r"((\d){3}-(\d){4})"
text = "This is a sample number 555-5555."
matches = re.findall(pattern, text)
print (matches)

[('555-5555', '5', '5')]


```[('555-5555', '5', '5')]```

'555-5555' - first capturing group

'5' - second capturing group, remembers only last digit in \d{3}

'5' - third capturing group
,remember last digit in \d{4}

In [5]:
#Import the requisite library
import spacy

#Sample text
text = "This is a sample number 5555555."
#Build upon the spaCy Small Model
nlp = spacy.blank("en")

#Create the Ruler and Add it
ruler = nlp.add_pipe("entity_ruler")


patterns = [
                {
                    "label": "PHONE_NUMBER", "pattern": [{"TEXT": {"REGEX": r"((\d){5})"}}
                                                        ]
                }
            ]
#add patterns to ruler
ruler.add_patterns(patterns)


#create the doc
doc = nlp(text)

#extract entities
for ent in doc.ents:
    print (ent.text, ent.label_)

5555555 PHONE_NUMBER


when we add a pattern like

``` {"pattern": [{"TEXT": {REGEX: (r"(\d){3}-(\d){4}"}}]```

a pattern wont be identified as spacy tokenizes hence no single match will be found as the match is broken into single components


In [11]:
text = "Paul Newman was an American actor, but Paul Hollywood is a British TV Host. The name Paul is quite common."
pattern = r"Paul(?: [A-Z]\w+)?" # gets names starting with 'Paul' and their surnames, or just 'Paul'
matches = re.finditer(pattern, text)

for match in matches:
  print(match)

<re.Match object; span=(0, 11), match='Paul Newman'>
<re.Match object; span=(39, 53), match='Paul Hollywood'>
<re.Match object; span=(85, 89), match='Paul'>


## Character Classes (Metacharacters)

Character classes allow you to match specific types of characters:

*   **`.`**: Matches any character (except newline by default, but this can be changed with flags like `re.DOTALL`).
    *   Example: `a.b` matches "axb", "a-b", "a1b", etc.
*   **`\d`**: Matches any digit (0-9).
    *   Equivalent to `[0-9]`.
*   **`\D`**: Matches any non-digit character.
    *   Equivalent to `[^0-9]`.
*   **`\w`**: Matches any word character (alphanumeric characters plus underscore: `a-zA-Z0-9_`).
    *   Equivalent to `[a-zA-Z0-9_]`.
*   **`\W`**: Matches any non-word character.
    *   Equivalent to `[^a-zA-Z0-9_]`.
*   **`\s`**: Matches any whitespace character (space, tab, newline, carriage return, form feed, vertical tab).
    *   Equivalent to `[ \t\n\r\f\v]`.
*   **`\S`**: Matches any non-whitespace character.
    *   Equivalent to `[^ \t\n\r\f\v]`.

# Reconstruct Spans

In [12]:
from spacy.tokens import Span

In [28]:
#prints an empty tuple since the pipeline is blank
pattern = r"Paul [A-Z]\w+"
nlp = spacy.blank("en")
doc = nlp(text)
print(doc.ents)
original_ents= list(doc.ents)#no entities will be present as it is a blank pipeline
multi_ents = []#multiword entities

for match in re.finditer(pattern, doc.text):
  start,end  = match.span() #returns span of where the full match occured
  span = doc.char_span(start,end)#returns a span object, moves from characters/letters to full words

  if span is not None:
    span_object = Span(doc, span.start,span.end,label = 'PERSON')
    original_ents.append(span_object)

doc.ents = original_ents #reassigning doc.ents
print(doc.ents)





()
(Paul Newman, Paul Hollywood)


In [29]:
for ent in doc.ents:
  print(ent.text, ent.label_)

Paul Newman PERSON
Paul Hollywood PERSON


In [31]:
from spacy.language import Language

@Language.component("paul_ner")
def paul_ner(doc):
  original_ents= list(doc.ents)#no entities will be present as it is a blank pipeline


  for match in re.finditer(pattern, doc.text):
    start,end  = match.span() #returns span of where the full match occured
    span = doc.char_span(start,end)#returns a span object, moves from characters/letters to full words

    if span is not None:
      span_object = Span(doc, span.start,span.end,label = 'PERSON')#has functioned as an entity ruler as it allows u to assign labels
      original_ents.append(span_object)

  doc.ents = original_ents #reassigning doc.ents
  return doc


# Workflow

A span is begginning and end of a token or sequence
I used match.span() where the pattern was identified . This retruns the span in character form.
I then converted the character span into token span which is done by the doc object using its doc.char_span() which takes character spans and converts them into token spans.  
Then i took the start and end of the spans to form the span object

In [35]:
nlp2 = spacy.blank("en")
nlp2.add_pipe("paul_ner")
doc2 = nlp2(text)
print(doc2.ents)


(Paul Newman, Paul Hollywood)


In [20]:
print(doc.text)

Paul Newman was an American actor, but Paul Hollywood is a British TV Host. The name Paul is quite common.


In [23]:
print(multi_ents)

[(0, 2, 'Paul Newman'), (8, 10, 'Paul Hollywood')]
